# 01 · Exploración del dataset de sedimentos

**Objetivo.** Entender la estructura del dataset ancho construido por
`run_pipeline.py`: cuántas muestras y puntos hay, qué parámetros tienen
cobertura suficiente y cómo se distribuyen las concentraciones.

> Requiere haber ejecutado `python run_pipeline.py --solo-datos`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"

## 1. Carga del dataset ancho y del dataset de riesgo

In [ ]:
ancho = pd.read_csv(PROCESSED / "dataset_ancho.csv", encoding="utf-8-sig")
riesgo = pd.read_csv(PROCESSED / "dataset_riesgo.csv", encoding="utf-8-sig")
print(f"ancho : {ancho.shape[0]} muestras x {ancho.shape[1]} columnas")
print(f"riesgo: {riesgo.shape[0]} muestras x {riesgo.shape[1]} columnas")
riesgo[["id_muestra", "fecha", "PUNTO_NORMALIZADO", "DISTRITO", "ANIO", "PLI", "RI", "nivel_riesgo"]].head()

## 2. Cobertura de parámetros

In [ ]:
cob = pd.read_csv(TABLES / "cobertura_parametros.csv", encoding="utf-8-sig")
print("Parámetros con cobertura >= 60 %:", int(cob["retenido_modelo"].sum()))
print("Parámetros con cobertura >= 10 %:", int(cob["retenido_analisis"].sum()))
cob.head(25)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
top = cob.head(30).iloc[::-1]
ax.barh(top["parametro"], top["cobertura"], color="#2c7bb6")
ax.axvline(0.6, color="#d7191c", ls="--", lw=1, label="umbral modelado (60 %)")
ax.axvline(0.1, color="#fdae61", ls=":", lw=1, label="umbral análisis (10 %)")
ax.set_xlabel("Cobertura (fracción de muestras con dato)")
ax.set_title("Cobertura de los 30 parámetros más completos")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura.** Los metales del índice (As, Cd, Cr, Cu, Hg, Ni, Pb, Zn) son los
parámetros con mayor cobertura, lo que justifica construir el PLI con ellos.
Los parámetros con cobertura baja (hidrocarburos, granulometría) se usan solo
como contexto y el modelo los imputa dentro del pipeline.

## 3. Distribución de los metales del índice

In [ ]:
from src import config as cfg

metales = cfg.METALES_INDICE
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, m in zip(axes.ravel(), metales):
    v = pd.to_numeric(riesgo[m], errors="coerce").dropna()
    ax.hist(np.log10(v.clip(lower=1e-4)), bins=25, color="#2c7bb6", edgecolor="white")
    ax.set_title(f"{m} (n={len(v)})", fontsize=9)
    ax.set_xlabel("log10(mg/kg)")
plt.suptitle("Distribución de los 8 metales del índice")
plt.tight_layout()
plt.show()

riesgo[metales].describe().T.round(3)

**Lectura.** Todas las concentraciones son asimétricas a la derecha, de ahí la
transformación `log1p` aplicada antes del modelado y del cálculo de distancias.

## 4. Cobertura espacial y temporal

In [ ]:
print(riesgo["DISTRITO"].value_counts())
print()
print(riesgo["ANIO"].value_counts().sort_index())
print()
print("Puntos normalizados:", riesgo["PUNTO_NORMALIZADO"].nunique())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
sc = ax.scatter(riesgo["COORD_ESTE"], riesgo["COORD_NORTE"],
                c=riesgo["PLI"], cmap="RdYlBu_r", s=28, edgecolor="k", linewidth=0.3)
fig.colorbar(sc, ax=ax, label="PLI")
ax.set_aspect("equal")
ax.set_xlabel("Este (UTM 19S)")
ax.set_ylabel("Norte (UTM 19S)")
ax.set_title("Distribución espacial de las muestras")
plt.tight_layout()
plt.show()

**Lectura.** La mayor densidad de muestras está en Carumas (cuenca alta), con
una cola de campañas recientes (2024-2026) que permite el hold-out temporal.

## 5. Conclusiones del capítulo

In [ ]:
resumen = {
    "muestras": len(riesgo),
    "puntos": riesgo["PUNTO_NORMALIZADO"].nunique(),
    "periodo": f"{int(riesgo['ANIO'].min())}-{int(riesgo['ANIO'].max())}",
    "distritos": riesgo["DISTRITO"].nunique(),
    "parametros_modelo": int(cob["retenido_modelo"].sum()),
    "PLI_mediano": round(float(riesgo["PLI"].median()), 2),
}
pd.Series(resumen, name="valor").to_frame()